In [1]:
import numpy as np
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
import time

features_size = 4;
sample_size = 100;

dataset = load_breast_cancer()

X = dataset.data
y = dataset.target
# feature_names = dataset.feature_names
# target_names = dataset.target_names

# 20 percent of data is used for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale from 0 to pi
scaler = MinMaxScaler(feature_range=(0, np.pi))

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Encode features into qubits
feature_map = zz_feature_map(
    feature_dimension = features_size,
    reps = 1,
    entanglement = "linear"
)

# Encode the first training sample
sample = X_train_scaled[0][:features_size]

encoded_circuit = feature_map.assign_parameters(sample)

print("Dataset:", X.shape)
print("Training data:", X_train.shape)
print("Test data:", X_test.shape)

print("Scaled training data:", X_train_scaled.shape)
print("Scaled test data:", X_test_scaled.shape)

print("Feature map qubits:", feature_map.num_qubits)
print("Feature map parameters:", feature_map.num_parameters)

print("Sample circuit:")
print(encoded_circuit.draw())

# Calculate quantum kernel
quantum_kernel = FidelityQuantumKernel(
    feature_map=feature_map
)

X_train_subset = X_train_scaled[:, :features_size]
X_test_subset = X_test_scaled[:, :features_size]


print("Selected training features:", X_train_subset.shape)
print("Selected test features:", X_test_subset.shape)

Dataset: (569, 30)
Training data: (455, 30)
Test data: (114, 30)
Scaled training data: (455, 30)
Scaled test data: (114, 30)
Feature map qubits: 4
Feature map parameters: 4
Sample circuit:
   ┌───┐┌────────────┐                                                  »
0: ┤ H ├┤ P(0.99293) ├──■─────────────────■─────────────────────────────»
   ├───┤├───────────┬┘┌─┴─┐┌───────────┐┌─┴─┐                           »
1: ┤ H ├┤ P(1.4109) ├─┤ X ├┤ P(12.888) ├┤ X ├──■────────────────■───────»
   ├───┤├───────────┴┐└───┘└───────────┘└───┘┌─┴─┐┌──────────┐┌─┴─┐     »
2: ┤ H ├┤ P(0.93438) ├───────────────────────┤ X ├┤ P(13.03) ├┤ X ├──■──»
   ├───┤├────────────┤                       └───┘└──────────┘└───┘┌─┴─┐»
3: ┤ H ├┤ P(0.48388) ├─────────────────────────────────────────────┤ X ├»
   └───┘└────────────┘                                             └───┘»
«                    
«0: ─────────────────
«                    
«1: ─────────────────
«                    
«2: ──────────────■──
«   ┌────────

In [2]:
start_kernel_train = time.time()

K_train = quantum_kernel.evaluate(
    x_vec=X_train_subset[:sample_size]
)

print("Training kernel time (seconds):", round(time.time() - start_kernel_train, 2))
print(K_train.shape)

Training kernel time (seconds): 9.37
(100, 100)


In [3]:
start_kernel_test = time.time()

K_test = quantum_kernel.evaluate(
    x_vec=X_test_subset[:sample_size],
    y_vec=X_train_subset[:sample_size]
)

print("Test kernel time (seconds):", round(time.time() - start_kernel_test, 2))
print(K_test.shape)

Test kernel time (seconds): 17.55
(100, 100)


In [4]:
qsvc = SVC(kernel="precomputed")
qsvc.fit(K_train, y_train[:sample_size])

y_pred = qsvc.predict(K_test)

print("Accuracy:", accuracy_score(y_test[:sample_size], y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test[:sample_size], y_pred))

print("Classification report:")
print(classification_report(
    y_test[:sample_size],
    y_pred,
    target_names=dataset.target_names,
    zero_division=0
))

print("Malignant recall:", recall_score(
    y_test[:sample_size],
    y_pred,
    pos_label=0,
    zero_division=0
))

Accuracy: 0.85
Confusion matrix:
[[24 10]
 [ 5 61]]
Classification report:
              precision    recall  f1-score   support

   malignant       0.83      0.71      0.76        34
      benign       0.86      0.92      0.89        66

    accuracy                           0.85       100
   macro avg       0.84      0.82      0.83       100
weighted avg       0.85      0.85      0.85       100

Malignant recall: 0.7058823529411765
